# Lab 1 — Data Ingestion (Course 1: Lakeflow Connect)
Lessons 1.1–1.9 · exercises `1a`–`1h` · schema `lab1`

**How this lab works**
- Run cells top to bottom (`Shift+Enter`). After a kernel restart, run again from the top.
- Cells marked `# TODO` contain `<FILL-IN>`: replace it with your code, then run the `check(...)` cell.
- Stuck? Ask Claude in the terminal: *"hint for 1c"*. Solutions are in `notebooks/solutions/` (try first!).
- 🖥 **Local** means open-source Spark + Delta on your Mac. ☁ **Databricks** shows the real syntax for the exam.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))          # lets the notebook import labutils
from labutils import get_spark, path, reset_lab, show, checkpoint, table_path, landing
from labutils.check import check
from pyspark.sql import functions as F
spark = get_spark()
reset_lab("lab1")

## 1a · Delta Lake basics (lesson 1.1)
A Delta table = **Parquet data files** + a **`_delta_log/`** folder (the transaction log).
Every write creates a new **version** → ACID, time travel, `DESCRIBE HISTORY`.

**Task:** create table `lab1.demo_delta (id INT, name STRING)` and insert 3 rows.

In [ ]:
# TODO 1a
spark.sql("""<FILL-IN 1>""")
spark.sql("""<FILL-IN 2>""")

In [ ]:
t = table_path("lab1", "demo_delta")
print("data files :", sorted(f for f in os.listdir(t) if f.endswith(".parquet")))
print("_delta_log :", sorted(os.listdir(t + "/_delta_log")))
show(spark.sql("DESCRIBE HISTORY lab1.demo_delta").select("version", "operation", "operationParameters"))

In [ ]:
check("1a")

## 1b · Batch ingestion with CTAS (lesson 1.2)
☁ Databricks:
```sql
CREATE TABLE historical_users_bronze AS
SELECT * FROM read_files('/Volumes/dbacademy_ecommerce/v01/raw/users-historical', format => 'parquet');
```
🖥 Local: `read_files()` is Databricks-only. Use the *query files by path* form instead:
``SELECT * FROM parquet.`<path>` `` (works on Databricks too).

Notice the files starting with `_` (like `_SUCCESS`) — Spark ignores them.

In [ ]:
users_path = path("users-historical")
print(sorted(os.listdir(users_path)))
show(spark.sql(f"SELECT * FROM parquet.`{users_path}` LIMIT 5"))

In [ ]:
# TODO 1b: create lab1.users_bronze_ctas from the Parquet files with CTAS
spark.sql(f"""
CREATE TABLE lab1.users_bronze_ctas AS
<FILL-IN 1>
""")

In [ ]:
check("1b")

### Managed vs external tables — watch what DROP does
Run the cell and read the output. Then answer: *after `DROP TABLE`, which table's files still exist?*

In [ ]:
import shutil
ext_location = os.path.abspath("../.lab_external/users_ext")
shutil.rmtree(ext_location, ignore_errors=True)
spark.sql(f"CREATE TABLE lab1.users_external LOCATION '{ext_location}' AS SELECT * FROM lab1.users_bronze_ctas")
spark.sql("CREATE TABLE lab1.users_managed AS SELECT * FROM lab1.users_bronze_ctas")
managed_location = table_path("lab1", "users_managed")
spark.sql("DROP TABLE lab1.users_external")
spark.sql("DROP TABLE lab1.users_managed")
print("external files still there? ", os.path.exists(ext_location + "/_delta_log"))
print("managed  files still there? ", os.path.exists(managed_location + "/_delta_log"))

## 1c · Incremental batch with COPY INTO (lesson 1.3)
☁ Databricks:
```sql
CREATE TABLE users_bronze_ci;                         -- empty, no schema needed
COPY INTO users_bronze_ci
  FROM '/Volumes/.../users'  FILEFORMAT = parquet
  COPY_OPTIONS ('mergeSchema' = 'true');
```
🖥 Local: `COPY INTO` is Databricks-only, so the lab has a look-alike: `copy_into(spark, table, folder, format, merge_schema=..., force=...)`.
It keeps the same rules: **idempotent** (already-loaded files are skipped), `force` reloads, new columns need `mergeSchema`.

`landing.land("users", n)` simulates new files arriving in a volume.

In [ ]:
from labutils.ingest import copy_into, reset_state
reset_state("lab1.users_bronze_ci")
vol = landing.reset("users")
landing.land("users", 2)
spark.sql("CREATE TABLE lab1.users_bronze_ci")    # empty table without a schema

In [ ]:
# TODO 1c
# 1) load the landed files into lab1.users_bronze_ci (format "parquet")
<FILL-IN 1>
# 2) run the SAME load again. How many rows does it insert? Why?
<FILL-IN 2>
# 3) land the next batch (it has a NEW column `country`) and load it.
landing.land("users", 1)
#    It fails with a schema mismatch -> fix it with the right option.
<FILL-IN 3>
print(spark.table("lab1.users_bronze_ci").count(), "rows")

In [ ]:
check("1c")

## 1d · Auto Loader idea: incremental streaming with a checkpoint (lesson 1.4)
☁ Databricks Auto Loader (Python):
```python
(spark.readStream.format("cloudFiles")
      .option("cloudFiles.format", "parquet")
      .option("cloudFiles.schemaLocation", checkpoint_path)
      .load("/Volumes/.../users")
 .writeStream.option("checkpointLocation", checkpoint_path)
      .trigger(availableNow=True)
      .toTable("users_bronze_stream"))
```
☁ Databricks SQL: `CREATE OR REFRESH STREAMING TABLE t AS SELECT * FROM STREAM read_files(...)`, then `REFRESH STREAMING TABLE t`.

🖥 Local: `cloudFiles` is Databricks-only, but Auto Loader is **built on Structured Streaming**. The plain
file source + a **checkpoint** gives the same core behaviour: each file is processed **once**.
`trigger(availableNow=True)` = process everything new, then stop (incremental batch).

In [ ]:
vol_stream = landing.reset("users_stream")
users_schema = "user_id STRING, user_first_touch_timestamp BIGINT, email STRING"

In [ ]:
# TODO 1d: complete the streaming read/write
def refresh_users_stream():
    (spark.readStream
        .format(<FILL-IN 1>)                      # local file format of the landed files
        .schema(users_schema)                   # streaming file sources need a schema
        .load(vol_stream)
     .writeStream
        .option("checkpointLocation", <FILL-IN 2>)    # use checkpoint("lab1", "users_stream")
        .trigger(availableNow=True)
        .toTable("lab1.users_bronze_stream")
     .awaitTermination())

In [ ]:
landing.land("users_stream", 1); refresh_users_stream()
print("after run 1:", spark.table("lab1.users_bronze_stream").count())
landing.land("users_stream", 1); refresh_users_stream()
print("after run 2:", spark.table("lab1.users_bronze_stream").count())
refresh_users_stream()   # nothing new arrived -> nothing to do
print("after run 3:", spark.table("lab1.users_bronze_stream").count(), " <- same: files are never read twice")

In [ ]:
check("1d")

## 1e · Metadata columns on ingest (lesson 1.5)
☁ Databricks SQL:
```sql
CREATE TABLE sales_bronze AS
SELECT *, _metadata.file_modification_time AS file_modification_time,
          _metadata.file_name AS source_file, current_timestamp() AS ingestion_time
FROM read_files('/Volumes/.../sales-csv', format => 'csv', sep => '|', header => true);
```
🖥 Local: same idea with the PySpark API (the course shows it as the Python equivalent).
The CSV files are separated by `|` and have a header row.

In [ ]:
# TODO 1e
sales_df = (spark.read
    .option("header", True)
    .option("sep", <FILL-IN 1>)
    .csv(path("sales-csv"))
    .select("*",
            <FILL-IN 2>,     # source_file            <- _metadata.file_name
            <FILL-IN 3>,     # file_modification_time <- _metadata.file_modification_time
            <FILL-IN 4>))    # ingestion_time         <- current timestamp
sales_df.write.mode("overwrite").saveAsTable("lab1.sales_bronze")

In [ ]:
show(spark.sql("SELECT source_file, count(*) AS total FROM lab1.sales_bronze GROUP BY source_file ORDER BY 1"))
check("1e")

## 1f · Rescuing malformed data (lesson 1.6)
Look at the raw file: two prices are written as `$945.00` — not a valid DOUBLE.

☁ Databricks: `read_files(..., schema => 'item_id STRING, name STRING, price DOUBLE', rescueddatacolumn => '_rescued_data')`
→ `price` is NULL and `_rescued_data` = `{"price":"$945.00","_file_path":"..."}`. Fix with
`coalesce(price, cast(replace(_rescued_data:price, '$', '') AS DOUBLE))`.

🖥 Local: open-source Spark has a cousin: **`columnNameOfCorruptRecord`** (keeps the *whole raw line*).

In [ ]:
show(spark.read.text(path("malformed")))
items_raw = (spark.read
    .schema("item_id STRING, name STRING, price DOUBLE, _corrupt_record STRING")
    .option("header", True)
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .csv(path("malformed"))
    .cache())
items_raw.createOrReplaceTempView("items_raw")
show(items_raw)

In [ ]:
# TODO 1f: fill price_fixed — use price, or the 3rd value of the raw line without "$"
spark.sql("""
CREATE OR REPLACE TABLE lab1.items_bronze AS
SELECT item_id, name, price,
       <FILL-IN 1> AS price_fixed,
       _corrupt_record
FROM items_raw
""")

In [ ]:
show(spark.table("lab1.items_bronze")); check("1f")

## 1g · Semi-structured JSON (lesson 1.7)
The Kafka-style events have **base64-encoded** `key` and `value` columns.
1. Decode: `cast(unbase64(value) AS STRING)` → a JSON-formatted STRING.
2. ☁ Databricks reads fields with colon syntax: `decoded_value:user_id`, `decoded_value:geo:city`.
   🖥 Local equivalent: `get_json_object(decoded_value, '$.user_id')`.
3. Convert to a STRUCT with `from_json(col, schema)`; find the schema with `schema_of_json`.
4. One row per array element with `explode`; **`explode_outer` keeps rows whose array is empty**.

In [ ]:
show(spark.read.json(path("events-kafka")), 3)

In [ ]:
# TODO 1g-1: decode the base64 value column
spark.sql(f"""
CREATE OR REPLACE TABLE lab1.events_decoded AS
SELECT key, offset, partition, timestamp, topic,
       <FILL-IN 1> AS decoded_value
FROM json.`{path('events-kafka')}`
""")

In [ ]:
sample = spark.table("lab1.events_decoded").first().decoded_value
print(sample)
print(spark.sql(f"SELECT schema_of_json('{sample}') AS s").first().s)
show(spark.sql("SELECT get_json_object(decoded_value, '$.user_id') AS user_id, "
               "get_json_object(decoded_value, '$.geo.city') AS city FROM lab1.events_decoded"), 3)

In [ ]:
# TODO 1g-2: parse to a STRUCT and flatten: one row per item, but KEEP events without items
event_schema = "<FILL-IN 1>"
spark.sql(f"""
CREATE OR REPLACE TABLE lab1.events_flat AS
WITH parsed AS (SELECT from_json(decoded_value, '{event_schema}') AS e FROM lab1.events_decoded),
     exploded AS (SELECT e, <FILL-IN 2>(e.items) AS item FROM parsed)
SELECT e.user_id, e.event_name, e.geo.city AS city, item.item_id, item.price_usd, item.quantity
FROM exploded
""")

In [ ]:
show(spark.table("lab1.events_flat"), 8); check("1g")

## 1h · MERGE INTO, schema evolution, time travel (lesson 1.9)
`MERGE INTO target USING source ON <key> WHEN MATCHED ... WHEN NOT MATCHED ...`
Source rows have a `status`: `update` → update the email, `delete` → delete the row, `new` → insert.

In [ ]:
spark.sql("""CREATE OR REPLACE TABLE lab1.main_users_target AS SELECT * FROM VALUES
  (1, 'Peter',   'peter@old.com',   DATE'2024-01-01', 'current'),
  (2, 'Zebi',    'zebi@x.com',      DATE'2024-01-02', 'current'),
  (3, 'Samarth', 'samarth@x.com',   DATE'2024-01-03', 'current')
  AS t(id, first_name, email, sign_up_date, status)""")
spark.sql("""CREATE OR REPLACE TABLE lab1.update_users_source AS SELECT * FROM VALUES
  (1, 'Peter',  'peter@new.com',  DATE'2024-01-01', 'update'),
  (2, 'Zebi',   'zebi@x.com',     DATE'2024-01-02', 'delete'),
  (4, 'Kostas', 'kostas@x.com',   DATE'2024-05-20', 'new')
  AS t(id, first_name, email, sign_up_date, status)""")

In [ ]:
# TODO 1h-1: write the MERGE
spark.sql("""
MERGE INTO lab1.main_users_target t
USING lab1.update_users_source s
ON <FILL-IN 1>
WHEN MATCHED AND s.status = 'update' THEN <FILL-IN 2>
WHEN MATCHED AND s.status = 'delete' THEN <FILL-IN 3>
WHEN NOT MATCHED THEN <FILL-IN 4>
""")
show(spark.sql("SELECT * FROM lab1.main_users_target ORDER BY id"))

**Schema evolution.** The next source has a new column `country`. Without evolution, MERGE fails
(schema enforcement). ☁ Databricks: `MERGE WITH SCHEMA EVOLUTION INTO ...` (DBR 15.2+), or set
`spark.databricks.delta.schema.autoMerge.enabled = true`. 🖥 Locally only the config works.

In [ ]:
spark.sql("""CREATE OR REPLACE TABLE lab1.new_users_source AS SELECT * FROM VALUES
  (5, 'Ana', 'ana@x.com', DATE'2024-06-01', 'new', 'CA')
  AS t(id, first_name, email, sign_up_date, status, country)""")

In [ ]:
# TODO 1h-2: enable schema evolution, then MERGE new users in (insert all columns)
<FILL-IN 1>
spark.sql("""
MERGE INTO lab1.main_users_target t
USING lab1.new_users_source s
ON t.id = s.id
WHEN NOT MATCHED THEN <FILL-IN 2>
""")
show(spark.sql("SELECT * FROM lab1.main_users_target ORDER BY id"))

In [ ]:
show(spark.sql("DESCRIBE HISTORY lab1.main_users_target").select("version", "operation"))
print("version 0 (before any MERGE):")
show(spark.sql("SELECT * FROM lab1.main_users_target VERSION AS OF 0 ORDER BY id"))
check("1h")

## Done 🎉
Ask Claude: `/quiz ingestion 5` to test yourself, or run the original course notebooks in Databricks Free Edition
(see `databricks/FREE_EDITION_GUIDE.md`) to try `read_files`, Auto Loader and streaming tables for real.